# Duka Electronics Sales Exploration

Half-year review | January–June 2026 | KES

## Executive Summary

- Nairobi led the 5 branches with KES 26,106,100 in revenue from 376 orders.
- Laptops contributed the most revenue at KES 41,535,000, while Tablets led unit sales with 691 units.
- Weekdays produced KES 51,856,200 across 723 orders; weekends produced KES 19,936,300 across 277 orders and had a KES 71,972.20 average order revenue.
- April was the strongest month at KES 13,518,300, 23.98% above June, the weakest month at KES 10,903,400.
- The Nairobi and Mombasa Premium Online segment contained 83 orders worth KES 10,218,000.

## Task 1 — First-Five-Minutes Inspection

Load the source with `order_date` parsed as a date, then inspect the shape, first and last records, column types, missingness, date coverage, distinct values, and core category distributions before drawing conclusions.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Dataset shape:", sales.shape)
print("\nFirst five rows:\n", sales.head().to_string(index=False))
print("\nLast five rows:\n", sales.tail().to_string(index=False))
print("\nMissing values by column:\n", sales.isna().sum().to_string())
print("\nDate range:", sales.order_date.min().date(), "to", sales.order_date.max().date())
print("\nColumn types:\n", sales.dtypes.to_string())
for column in ["branch", "product", "channel"]:
    print(f"\n{column.title()} counts:\n{sales[column].value_counts().to_string()}")
sales.head()

Dataset shape: (1000, 7)

First five rows:
  order_id order_date  branch    product  quantity  channel  unit_price
     1001 2026-06-03  Nakuru      Phone         3 In-store       15000
     1002 2026-02-02 Nairobi     Laptop         4 In-store       65000
     1003 2026-01-05  Nakuru     Tablet         5 In-store       28000
     1004 2026-04-26 Nairobi Headphones         5 In-store        2500
     1005 2026-03-07 Nairobi Headphones         3 In-store        2500

Last five rows:
  order_id order_date  branch    product  quantity     channel  unit_price
     1996 2026-03-04 Mombasa     Laptop         5    In-store       65000
     1997 2026-03-06 Nairobi     Tablet         2 Phone order       28000
     1998 2026-01-20 Nairobi     Laptop         3      Online       65000
     1999 2026-03-02 Nairobi Headphones         2    In-store        2500
     2000 2026-01-30  Nakuru    Charger         4    In-store        1200

Missing values by column:
 order_id      0
order_date    0
branch  

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


### First-Five-Minutes Findings

- **Shape:** 1,000 order records and 7 source columns.
- **Types:** IDs, quantities, and prices are integers; `order_date` is a parsed datetime; branch, product, and channel are categorical text.
- **Missingness:** No missing values were found in any source column.
- **Date range:** January 1 through June 29, 2026, covering the requested half-year period.
- **Categories:** There are 5 branches, 5 products, and 3 channels. Nairobi has the most orders (376); In-store is the most common channel (579).
- **Sanity check:** Quantities run from 1 to 5; unit prices span KES 1,200 to KES 65,000.

## Task 2 — Derived Columns

Create `revenue` as quantity × unit price; derive calendar `month` and `weekday`; flag Saturday/Sunday in `is_weekend`; and classify unit prices of at least KES 28,000 as Premium, otherwise Value.

In [2]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")
print("Derived columns preview:\n", sales.head().to_string(index=False))
print("\nDerived columns:", ["revenue", "month", "weekday", "is_weekend", "price_band"])

Derived columns preview:
  order_id order_date  branch    product  quantity  channel  unit_price  revenue    month   weekday  is_weekend price_band
     1001 2026-06-03  Nakuru      Phone         3 In-store       15000    45000     June Wednesday       False      Value
     1002 2026-02-02 Nairobi     Laptop         4 In-store       65000   260000 February    Monday       False    Premium
     1003 2026-01-05  Nakuru     Tablet         5 In-store       28000   140000  January    Monday       False    Premium
     1004 2026-04-26 Nairobi Headphones         5 In-store        2500    12500    April    Sunday        True      Value
     1005 2026-03-07 Nairobi Headphones         3 In-store        2500     7500    March  Saturday        True      Value

Derived columns: ['revenue', 'month', 'weekday', 'is_weekend', 'price_band']


C:\Users\gkisi\AppData\Local\Temp\ipykernel_20648\1012852519.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5


## Task 3 — Branch Rankings

Rank each branch by total revenue and by order count.

In [3]:
branch_revenue = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)
branch_orders = sales.groupby("branch")["order_id"].count().sort_values(ascending=False)
print("Branch ranking by total revenue (KES):\n", branch_revenue.to_string())
print("\nBranch ranking by number of orders:\n", branch_orders.to_string())

Branch ranking by total revenue (KES):
 branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200

Branch ranking by number of orders:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117


### Answer

The rankings are identical: Nairobi (KES 26,106,100; 376 orders), Mombasa (KES 14,003,700; 186), Nakuru (KES 11,970,400; 162), Kisumu (KES 10,868,100; 159), and Eldoret (KES 8,844,200; 117). In this dataset, the greater order counts at each branch also correspond to greater revenue totals; per-order value differences did not reverse any branch's position.

## Task 4 — Top Products by Revenue and Units Sold

Compare the three highest-revenue products with the three products having the most units sold.

In [4]:
product_revenue = sales.groupby("product")["revenue"].sum().sort_values(ascending=False)
product_units = sales.groupby("product")["quantity"].sum().sort_values(ascending=False)
print("Top products by revenue (KES):\n", product_revenue.head(3).to_string())
print("\nTop products by units sold:\n", product_units.head(3).to_string())

Top products by revenue (KES):
 product
Laptop    41535000
Tablet    19348000
Phone      8880000

Top products by units sold:
 product
Tablet    691
Laptop    639
Phone     592


### Answer

The top three by revenue are Laptop (KES 41,535,000), Tablet (KES 19,348,000), and Phone (KES 8,880,000). By units sold they are Tablet (691), Laptop (639), and Phone (592). The membership is the same, but Laptop and Tablet swap first and second place: the Laptop's higher unit price generates more revenue despite fewer units sold.

## Task 5 — Weekend vs Weekday Orders

Compare order count, total revenue, and mean order revenue for Saturday/Sunday versus Monday–Friday.

In [5]:
weekend_comparison = sales.groupby("is_weekend").agg(
    number_of_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean"),
)
weekend_comparison.index = weekend_comparison.index.map({False: "Weekday", True: "Weekend"})
print(weekend_comparison.to_string(formatters={"average_order_revenue": "{:,.2f}".format}))

            number_of_orders  total_revenue average_order_revenue
is_weekend                                                       
Weekday                  723       51856200             71,723.65
Weekend                  277       19936300             71,972.20


### Answer

Weekdays generated much more aggregate business: 723 orders and KES 51,856,200, versus 277 orders and KES 19,936,300 on weekends. The mean revenue per order was slightly higher on weekends (KES 71,972.20) than weekdays (KES 71,723.65), a difference of about KES 248.55 per order.

## Task 6 — Monthly Revenue

Identify the best and worst months and calculate how far the best month exceeded the worst, using the worst month as the percentage denominator.

In [6]:
monthly_revenue = sales.groupby("month")["revenue"].sum().sort_values(ascending=False)
best_month, worst_month = monthly_revenue.idxmax(), monthly_revenue.idxmin()
best_revenue, worst_revenue = monthly_revenue.max(), monthly_revenue.min()
percentage_difference = (best_revenue - worst_revenue) / worst_revenue * 100
print("Revenue by month (KES):\n", monthly_revenue.to_string())
print(f"\nBest month: {best_month} — KES {best_revenue:,}")
print(f"Worst month: {worst_month} — KES {worst_revenue:,}")
print(f"Percentage difference (relative to worst): {percentage_difference:.2f}%")

Revenue by month (KES):
 month
April       13518300
March       12396800
January     11786600
February    11772000
May         11415400
June        10903400

Best month: April — KES 13,518,300
Worst month: June — KES 10,903,400
Percentage difference (relative to worst): 23.98%


### Answer

April was best at KES 13,518,300; June was worst at KES 10,903,400. April's revenue was 23.98% above June's, calculated as (April − June) ÷ June × 100.

## Task 7 — Premium Online Orders: Nairobi and Mombasa

Filter to Premium price band, Online channel, and the Nairobi or Mombasa branch; report count and revenue, then export without the index.

In [7]:
premium_online = sales.loc[
    sales["price_band"].eq("Premium")
    & sales["channel"].eq("Online")
    & sales["branch"].isin(["Nairobi", "Mombasa"])
].copy()
Path("outputs").mkdir(exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print(f"Matching orders: {len(premium_online)}")
print(f"Total revenue: KES {premium_online['revenue'].sum():,}")
print("Export written: outputs/premium_online_coast_capital.csv (index=False)")
print("\nFirst five matching orders:\n", premium_online.head().to_string(index=False))

Matching orders: 83
Total revenue: KES 10,218,000
Export written: outputs/premium_online_coast_capital.csv (index=False)

First five matching orders:
  order_id order_date  branch product  quantity channel  unit_price  revenue    month   weekday  is_weekend price_band
     1007 2026-01-15 Nairobi  Laptop         2  Online       65000   130000  January  Thursday       False    Premium
     1014 2026-06-12 Nairobi  Tablet         3  Online       28000    84000     June    Friday       False    Premium
     1033 2026-04-29 Nairobi  Laptop         2  Online       65000   130000    April Wednesday       False    Premium
     1036 2026-03-02 Nairobi  Laptop         4  Online       65000   260000    March    Monday       False    Premium
     1040 2026-02-10 Mombasa  Laptop         5  Online       65000   325000 February   Tuesday       False    Premium


### Answer

The filter returns 83 orders worth KES 10,218,000 in total revenue. All exported rows meet the Premium, Online, and Nairobi-or-Mombasa conditions; the CSV is saved at `outputs/premium_online_coast_capital.csv` with `index=False`.

## Task 8 — Spreadsheet Reconciliation

The workbook `excel/duka_branch_summary.xlsx` contains a SUMIFS branch-revenue table, XLOOKUP-derived region, and a PivotTable of revenue by branch and channel. The check below compares the five branch totals and grand total against the pandas calculations.

In [8]:
excel_branch_revenue = {
    "Nairobi": 26106100,
    "Mombasa": 14003700,
    "Nakuru": 11970400,
    "Kisumu": 10868100,
    "Eldoret": 8844200,
}
pandas_branch_revenue = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)
excel_total = sum(excel_branch_revenue.values())
pandas_total = sales["revenue"].sum()
print("Pandas branch revenue (KES):\n", pandas_branch_revenue.to_string())
print("\nExcel branch totals match pandas:", all(excel_branch_revenue[b] == pandas_branch_revenue[b] for b in excel_branch_revenue))
print(f"Excel PivotTable grand total: KES {excel_total:,}")
print(f"Pandas total revenue:          KES {pandas_total:,}")
print("Grand totals match:", excel_total == pandas_total)

Pandas branch revenue (KES):
 branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200

Excel branch totals match pandas: True
Excel PivotTable grand total: KES 71,792,500
Pandas total revenue:          KES 71,792,500
Grand totals match: True


### Spreadsheet Result

The spreadsheet and pandas branch rankings agree for all 5 branches. The PivotTable grand total is KES 71,792,500, equal to the pandas total revenue, so the spreadsheet reconciliation passes.